# Session 00 — From Language Models to GenAI Applications

This is a map for the course, not a compressed replacement for Sessions 09–12. By the end, you should be able to explain the major model families, trace text into vectors, compare model-access options, and decide when RAG or an agent is justified.

> **Run safely:** offline cells run without credentials. Provider cells are opt-in and read keys from environment variables. Never paste a key into a notebook.

## Notebook setup: local Jupyter or Google Colab

Run the next two cells before the provider examples. The installer targets the current notebook kernel. In Colab, upload your `.env` file into the notebook's working directory using the **Files** panel. Local learners can keep `.env` at the repository or notebook working directory.

> The Hugging Face example needs `transformers` and `torch`. Colab often already has `torch`; pip will reuse a compatible installation when possible.

In [ ]:
import sys

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    !{sys.executable} -m pip install -q python-dotenv httpx openai langchain-core langchain-openai transformers torch
else:
    # The course's uv environment intentionally has no pip; install into this exact kernel.
    !uv pip install --python {sys.executable} python-dotenv httpx openai langchain-core langchain-openai transformers torch

In [ ]:
import os
from pathlib import Path

from dotenv import find_dotenv, load_dotenv

env_path = find_dotenv(usecwd=True)
if env_path:
    load_dotenv(env_path, override=False)
    print(f"Loaded environment variables from {Path(env_path).name}")
else:
    print("No .env file found. Offline cells still work; upload or create .env for provider calls.")

configured = [name for name in ("OPENAI_API_KEY", "OPENROUTER_API_KEY") if os.getenv(name)]
print("Configured provider variables:", configured or "none")  # names only; never print values

## 1. The course bridge

![Course bridge roadmap](../../../assets/diagrams/course-bridge-roadmap.svg)

**How this works**

- Your existing design, API, testing and debugging experience remains the foundation.
- Python fluency comes before AI frameworks, so abstractions remain debuggable.
- HTTP, validation, FastAPI and async are ordinary application engineering used around model calls.
- The capstone integrates retrieval, an LLM, a FastAPI boundary and a Streamlit UI.

## 2. How language models became large language models

A language model estimates what text is likely to come next. Earlier statistical models counted short sequences. Neural models learned representations but struggled with long dependencies and sequential training. The Transformer replaced recurrence with attention: each token can weigh relevant tokens, and training can process many positions in parallel.

Three forces then reinforced one another:

1. **Architecture:** Transformers scaled effectively and supported parallel training.
2. **Hardware and systems:** GPUs/accelerators, larger memory, faster interconnects and distributed training made far larger runs practical.
3. **Data and engineering:** large corpora, better tokenisation, training recipes and post-training turned predictors into useful assistants.

`Large` is not a precise threshold. It usually signals scale in parameters, training data and compute, plus broad capabilities that emerge from that combination.

## 3. Words → tokens → token IDs → embedding vectors

![From text to vectors](../../../assets/diagrams/text-to-vector.svg)

**How this works**

- A tokenizer splits text into model-specific units. A token can be a whole word, a word fragment, punctuation, whitespace or bytes.
- The tokenizer's **vocabulary** is a finite lookup table from token text to integer token ID. That is the model's input alphabet—not a dictionary of English words.
- Token IDs are indices, not quantities: ID 900 is not semantically ‘larger’ than ID 12.
- Each ID selects a row from a learned embedding table, producing a dense vector.
- Position information and Transformer layers turn initial token embeddings into contextual representations. The same token can behave differently in different sentences.

In [ ]:
# A deliberately tiny vocabulary illustrates lookup; real tokenizers learn much larger vocabularies.
vocabulary = {"un": 1, "believ": 2, "able": 3}
embedding_table = {
    1: [0.12, -0.44, 0.08],
    2: [0.71, 0.08, -0.31],
    3: [-0.09, 0.35, 0.62],
}
tokens = ["un", "believ", "able"]
token_ids = [vocabulary[token] for token in tokens]
vectors = [embedding_table[token_id] for token_id in token_ids]
list(zip(tokens, token_ids, vectors))

**Predict → Run → Modify → Explain**

Before running, predict whether changing an ID changes its vector. Then add punctuation as a fourth token. Explain why an application must use the tokenizer paired with its model.

## 4. Base → instruct/chat → reasoning → hybrid

![Model family progression](../../../assets/diagrams/model-evolution.svg)

**How this works**

- A **base model** is trained primarily to continue token sequences; it is not automatically a cooperative assistant.
- An **instruction/chat model** is post-trained to follow instructions, converse and respect preferred behavior.
- A **reasoning model** is optimized to spend additional inference work on multi-step problems. More work can improve difficult tasks but adds latency and cost.
- A **hybrid** or configurable-reasoning model can operate in faster and more deliberate modes. Terminology varies by provider.
- These categories describe behavior and training intent; they are not a universal type system.

### Optional: observe raw completion with a tiny Hugging Face base model

This downloads the public `sshleifer/tiny-gpt2` model without a Hugging Face token. It is intentionally tiny and useful for mechanics—not answer quality. Install optional packages in the course environment with `uv pip install transformers torch`, then restart the kernel.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

model_id = "sshleifer/tiny-gpt2"
tokenizer = AutoTokenizer.from_pretrained(model_id)
base_model = AutoModelForCausalLM.from_pretrained(model_id)
prompt = "An incident postmortem should include"
inputs = tokenizer(prompt, return_tensors="pt")
generated = base_model.generate(**inputs, max_new_tokens=25, do_sample=False)
print(tokenizer.decode(generated[0], skip_special_tokens=True))

**Engineering Insight:** `generate()` continues a sequence. A chat template, system role and helpful assistant behavior are additional conventions or post-training—not properties of next-token prediction itself.

## 5. Model as a Service: REST first

A hosted LLM is still a remote dependency: authenticate, serialize a request, set a timeout, parse an imperfect response, record usage and handle failure. The OpenAI Responses API example below is opt-in.

In [ ]:
import os

import httpx

api_key = os.environ["OPENAI_API_KEY"]
payload = {
    "model": os.getenv("OPENAI_MODEL", "gpt-6-astra"),
    "input": "Explain idempotency to an experienced engineer in two sentences.",
}
response = httpx.post(
    "https://api.openai.com/v1/responses",
    headers={"Authorization": f"Bearer {api_key}"},
    json=payload,
    timeout=30.0,
)
response.raise_for_status()
data = response.json()
texts = [part["text"] for item in data["output"] if item["type"] == "message"
         for part in item["content"] if part["type"] == "output_text"]
print("\n".join(texts))

### OpenRouter: an OpenAI-compatible gateway

Free-model availability changes. Keep the model in `OPENROUTER_MODEL`; `openrouter/free` lets the gateway select from its current free pool. Treat free capacity as suitable for learning, not a production SLA.

In [ ]:
import os

from openai import OpenAI

openrouter = OpenAI(
    api_key=os.environ["OPENROUTER_API_KEY"],
    base_url="https://openrouter.ai/api/v1",
    timeout=30.0,
)
reply = openrouter.chat.completions.create(
    model=os.getenv("OPENROUTER_MODEL", "openrouter/free"),
    messages=[{"role": "user", "content": "Give one practical use for Pydantic."}],
)
print(reply.choices[0].message.content)

### Ollama: the same boundary, hosted on your machine

Install Ollama separately and pull a model, for example `ollama pull gemma3:1b`. The HTTP call requires no provider key; model files and compute remain local.

In [ ]:
import os

import httpx

ollama_response = httpx.post(
    "http://localhost:11434/api/chat",
    json={
        "model": os.getenv("OLLAMA_MODEL", "gemma3:1b"),
        "messages": [{"role": "user", "content": "What makes an API idempotent?"}],
        "stream": False,
    },
    timeout=60.0,
)
ollama_response.raise_for_status()
print(ollama_response.json()["message"]["content"])

### What LangChain abstracts

LangChain gives model wrappers, message objects, prompt templates, composition, structured-output helpers and integrations. It reduces repeated plumbing; it does not remove authentication, timeouts, provider capability differences, evaluation or operational ownership.

In [ ]:
import os

from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI

prompt = ChatPromptTemplate.from_messages([
    ("system", "Answer for an experienced software engineer in at most two sentences."),
    ("human", "{question}"),
])
model = ChatOpenAI(model=os.getenv("OPENAI_MODEL", "gpt-6-astra"), timeout=30)
chain = prompt | model
result = chain.invoke({"question": "Why validate LLM output?"})
print(result.content)

## 6. Why RAG emerged

Model weights are a poor place for frequently changing private knowledge. A model may also produce plausible statements without evidence. Retrieval-augmented generation (RAG) searches an external knowledge collection and adds selected passages to the request context. It changes the input, not the model weights.

![RAG similarity search](../../../assets/diagrams/rag-similarity-search.svg)

**How this works**

- Documents are split into chunks and embedded; each stored record keeps text, metadata and a vector.
- The question is embedded with a compatible embedding model.
- Cosine similarity ranks a chunk vector `d` against query vector `q`: `cos(q,d) = (q·d) / (||q|| ||d||) = cos θ`.
- The dot product measures alignment; dividing by both magnitudes makes the score about direction rather than vector length.
- The best chunks become evidence in the LLM context. A high score is not proof of relevance, so retrieval requires evaluation.

In [ ]:
from math import sqrt


def dot(left: list[float], right: list[float]) -> float:
    return sum(a * b for a, b in zip(left, right, strict=True))

def magnitude(vector: list[float]) -> float:
    return sqrt(dot(vector, vector))

def cosine_similarity(left: list[float], right: list[float]) -> float:
    denominator = magnitude(left) * magnitude(right)
    if denominator == 0:
        raise ValueError("Cosine similarity is undefined for a zero vector")
    return dot(left, right) / denominator

query = [1.0, 1.0]
chunks = {"same direction": [2.0, 2.0], "related": [1.0, 0.4], "opposite": [-1.0, -1.0]}
sorted(((name, cosine_similarity(query, vector)) for name, vector in chunks.items()),
       key=lambda item: item[1], reverse=True)

**Predict → Run → Modify → Explain:** Why do `[1, 1]` and `[100, 100]` score `1`? Add an orthogonal vector `[1, -1]`. Then explain why production retrieval also considers metadata, chunking, filters and top-k—not cosine similarity alone.

## 7. RAG, agents and agentic RAG

| Pattern | The application decides | The model decides | Prefer it when |
|---|---|---|---|
| Direct LLM | prompt and one call | generated response | knowledge is already in context |
| RAG | retrieve, then generate | answer from supplied evidence | external knowledge is required |
| Deterministic workflow | steps and branches | bounded content or classifications | the path is known |
| Agent | tools, limits and goal | next action in a loop | the path cannot be fixed cheaply |
| Agentic RAG | available retrieval tools and controls | when/how to search, refine or stop | retrieval needs adaptive planning |

An agent is not automatically more capable in production. Each decision adds latency, cost and another failure path. Start with the simplest architecture that meets the requirement.

## 8. Loop engineering and harness engineering

These are useful emerging terms, not rigid standards.

- **Loop engineering** designs the repeated cycle: observe state → choose an action → execute → inspect the result → continue or stop. It covers state transitions, planning policy, reflection and termination.
- **Harness engineering** builds the surrounding control system: typed tools, permissions, isolation, timeouts, budgets, retries, logging, tracing, evaluations, checkpoints and human approval.
- A strong model inside a weak harness can still leak data, loop forever or take the wrong action.
- Experienced engineers will recognise familiar ideas from workflow engines, distributed systems, testing and security boundaries.

## 9. What matters in production

Treat GenAI as probabilistic software inside a deterministic system. Define contracts, validate output, set time and cost limits, test representative cases, trace decisions, protect data, handle partial failure and decide where humans retain authority.

### Key takeaways

- LLMs predict tokens; assistant behavior and application reliability come from additional layers.
- Tokens map to IDs and initial vectors; contextual and document embeddings serve different purposes.
- REST, SDKs and LangChain are layers over model access, each with trade-offs.
- RAG supplies evidence. Agents choose actions. Agentic RAG makes retrieval adaptive.
- The loop creates behavior; the harness makes that behavior governable.